# Clapperboard detection experiment

Этот ноутбук загружает файлы из папки `claps` и пробует модель `wsntxxn/cnn8rnn-audioset-sed` для поиска хлопков / хлопушки. Он выводит метки, индексы и предполагаемые временные метки событий.


In [ ]:
# Dependencies are managed in the project .venv via uv.
# This notebook expects torch, torchaudio, transformers, soundfile, and ipywidgets to be installed there.


In [1]:
# Import modules and define helpers
from pathlib import Path

import torch
import torchaudio
from transformers import AutoModel


def load_mono_audio(path, target_sr):
    wav, sr = torchaudio.load(path)
    if sr != target_sr:
        wav = torchaudio.functional.resample(wav, sr, target_sr)
    if wav.size(0) > 1:
        wav = wav.mean(dim=0)
    else:
        wav = wav[0]
    return wav


def detect_events(frame_probs, frame_duration_s, threshold=0.4, min_duration_s=0.02):
    mask = frame_probs > threshold
    if mask.numel() == 0:
        return []
    edges = torch.nonzero(mask[1:] != mask[:-1], as_tuple=False).squeeze(-1) + 1
    if mask[0]:
        edges = torch.cat([torch.tensor([0], device=mask.device), edges])
    if mask[-1]:
        edges = torch.cat([edges, torch.tensor([mask.size(0)], device=mask.device)])
    events = []
    for start, end in edges.view(-1, 2):
        duration = (end - start) * frame_duration_s
        if duration >= min_duration_s:
            events.append(start.item() * frame_duration_s)
    return events


model_name = "wsntxxn/cnn8rnn-audioset-sed"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModel.from_pretrained(model_name, trust_remote_code=True).to(device)
print("Loaded model:", model_name)
print("Sample rate:", model.config.sample_rate)
print("Hop size:", getattr(model.config, "hop_size", None))
print("Number of classes:", len(model.classes))

for label in ["Clapping", "Bang", "Clapperboard", "Clap", "Clapping, handclapping"]:
    if label in model.classes:
        print("Found label:", label)


Loaded model: wsntxxn/cnn8rnn-audioset-sed
Sample rate: 32000
Hop size: None
Number of classes: 447
Found label: Clapping
Found label: Bang


In [2]:
# Batch inference on all WAV files in the claps folder
claps_dir = Path("claps")
files = sorted(claps_dir.glob("*.wav"))

if not files:
    raise FileNotFoundError(f"No WAV files found in {claps_dir.resolve()}")

sample_rate = model.config.sample_rate

print(f"Evaluating {len(files)} files from: {claps_dir.resolve()}")

for file_path in files:
    wav = load_mono_audio(str(file_path), sample_rate).to(device)
    wav = wav.unsqueeze(0)
    with torch.no_grad():
        output = model(waveform=wav)

    framewise = output["framewise_output"]
    frame_duration_s = (wav.shape[-1] / sample_rate) / framewise.shape[1]
    class_names = list(model.classes)

    for candidate_label in ["Clapping", "Clapperboard", "Clap", "Clapping, handclapping"]:
        if candidate_label in class_names:
            label_idx = class_names.index(candidate_label)
            probs = framewise[0, :, label_idx].cpu()
            timestamps = detect_events(probs, frame_duration_s, threshold=0.35)
            if timestamps:
                print(f"{file_path.name} -> {candidate_label}: {timestamps}")
    else:
        print(f"{file_path.name}: candidate label search done.")


Evaluating 7 files from: /Users/johnwunderbellen/Audio_scene_cutter_renamer/claps
ZOOM0398_Tr3 [2026-05-28 152751].wav: candidate label search done.
ZOOM0746_Tr3 [2026-05-28 152751].wav: candidate label search done.
ZOOM1473_Tr3 [2026-05-28 152751].wav: candidate label search done.
ZOOM1857_Tr3 [2026-05-28 152751].wav: candidate label search done.
ZOOM1858_Tr3 [2026-05-28 152751].wav: candidate label search done.
Тест_шепот.wav: candidate label search done.
тест.wav: candidate label search done.


In [3]:
timestamps

[]

In [4]:
class_names

['Accelerating, revving, vroom',
 'Air brake',
 'Air conditioning',
 'Air horn, truck horn',
 'Aircraft',
 'Aircraft engine',
 'Alarm',
 'Alarm clock',
 'Alert',
 'Ambulance (siren)',
 'Animal',
 'Applause',
 'Arrow',
 'Artillery fire',
 'Audio logo',
 'Babbling',
 'Baby cry, infant cry',
 'Baby laughter',
 'Background noise',
 'Bang',
 'Bark',
 'Basketball bounce',
 'Bathroom sounds',
 'Bathtub (filling or washing)',
 'Battle cry',
 'Bee, wasp, etc.',
 'Beep, bleep',
 'Bell',
 'Bellow',
 'Belly laugh',
 'Bicycle bell',
 'Bicycle, tricycle',
 'Bird',
 'Bird flight, flapping wings',
 'Bird vocalization, bird call, bird song',
 'Biting',
 'Bleat',
 'Blender, food processor',
 'Boat, Water vehicle',
 'Boiling',
 'Boing',
 'Booing',
 'Boom',
 'Bouncing',
 'Bow-wow',
 'Breaking',
 'Breathing',
 'Brief tone',
 'Burping, eructation',
 'Burst, pop',
 'Bus',
 'Busy signal',
 'Buzz',
 'Buzzer',
 'Cacophony',
 'Camera',
 'Canidae, wild dogs, wolves',
 'Cap gun',
 'Car',
 'Car alarm',
 'Car passin

In [ ]:
import torch
from transformers import AutoModel
import torchaudio

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModel.from_pretrained(
    "wsntxxn/cnn8rnn-audioset-sed",
    trust_remote_code=True
).to(device)

wav1, sr1 = torchaudio.load("claps/Тест_шепот.wav")
wav1 = torchaudio.functional.resample(wav1, sr1, model.config.sample_rate)
wav1 = wav1.mean(0) if wav1.size(0) > 1 else wav1[0]

wav2, sr2 = torchaudio.load("claps/ZOOM0759_Tr3.WAV")
wav2 = torchaudio.functional.resample(wav2, sr2, model.config.sample_rate)
wav2 = wav2.mean(0) if wav2.size(0) > 1 else wav2[0]

wav_batch = torch.nn.utils.rnn.pad_sequence([wav1, wav2], batch_first=True)

with torch.no_grad():
    output = model(waveform=wav_batch)
    # output: {
    #     "framewise_output": (2, 447, n_frames),
    #     "clipwise_output": (2, 447)
    # }

# classes is in `model.classes`
# for example, the probability sequence of male speech is:
male_speech_prob = output["framewise_output"][:, model.classes.index("Male speech, man speaking"), :]


KeyError: (slice(None, None, None), 238, slice(None, None, None))